In [2]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pytesseract
import os
import pandas as pd
import pytesseract
from PIL import Image

In [6]:
import os
import pandas as pd
from PIL import Image
import pytesseract

# ==========================================
# CHANGE ONLY THESE
# ==========================================

START_ID = 1
END_ID = 35

# ==========================================
# FILE PATHS
# ==========================================

folder = "/content/drive/MyDrive/Flood_Dataset/images"
csv = "/content/drive/MyDrive/Flood_Dataset/dataset.csv"

# ==========================================
# CSV COLUMNS
# ==========================================

columns = [
    "Id",
    "Source",
    "Text",
    "Image",
    "Extracted Text",
    "Language",
    "Code-Mixed",
    "Translated Text"
]

# ==========================================
# LOAD CSV
# ==========================================

if os.path.exists(csv):
    df = pd.read_csv(csv)
else:
    df = pd.DataFrame(columns=columns)

# ==========================================
# FIX COLUMNS
# ==========================================

for col in columns:
    if col not in df.columns:
        df[col] = ""

# IMPORTANT:
# Remove old Yes/No values from Text column
# because Text now stores actual social-media text.

df["Text"] = df["Text"].replace(
    ["Yes", "No", "yes", "no"],
    ""
)

# Make sure columns are in correct order
df = df[columns]

# ==========================================
# FIND IMAGES
# ==========================================

files = sorted([
    f for f in os.listdir(folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Images already processed
done_images = set(df["Image"].dropna().astype(str))

# ==========================================
# OCR
# ==========================================

current_id = START_ID

for file in files:

    if current_id > END_ID:
        break

    if file in done_images:
        continue

    image_path = os.path.join(folder, file)

    try:
        image = Image.open(image_path)

        extracted_text = pytesseract.image_to_string(
            image,
            lang="eng+hin+asm"
        ).strip()

        new_row = {
            "Id": current_id,
            "Source": "X",
            "Text": "",
            "Image": file,
            "Extracted Text": extracted_text,
            "Language": "",
            "Code-Mixed": "",
            "Translated Text": ""
        }

        df.loc[len(df)] = new_row

        print(f"Processed: {file} → ID {current_id}")

        current_id += 1

    except Exception as e:
        print(f"Error processing {file}: {e}")

# ==========================================
# SORT
# ==========================================

df = df.sort_values("Id")

# ==========================================
# SAVE
# ==========================================

df.to_csv(
    csv,
    index=False,
    encoding="utf-8-sig"
)

print("\nDataset saved successfully!")
print("Text column has been changed to actual text field.")
print("Columns:", list(df.columns))

display(df)


Dataset saved successfully!
Text column has been changed to actual text field.
Columns: ['Id', 'Source', 'Text', 'Image', 'Extracted Text', 'Language', 'Code-Mixed', 'Translated Text']


,Id,Source,Text,Image,Extracted Text,Language,Code-Mixed,Translated Text
0,1,NaN,,img1.jpeg,এতিয়া প্ৰাণটোৰ বাহিৰে নিজৰ\nবুলিবলৈ যেন একোৱে...,NaN,NaN,NaN
1,2,NaN,,img2.jpeg,"४० !\nay\n\nGa COMO\nকাপোৰ, তিতি যায় সামঞ্জ",NaN,NaN,NaN
